# Aves Classification — Google Colab Training

**Steps:**
1. Runtime → Change runtime type → select **T4 GPU**
2. Run cells in order
3. Upload the `nabirds/` folder to Google Drive (expected structure below)

## 1. GPU Check

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found! Go to Runtime > Change runtime type > T4 GPU.')

## 2. Mount Google Drive

**Upload your data to Drive first:**
- Upload the `data/nabirds/` folder to your Google Drive
- Expected structure: `MyDrive/nabirds/images/`, `MyDrive/nabirds/train.csv`, `MyDrive/nabirds/val.csv`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2.5. (Optional) Copy Data to Colab Local Disk

**Skip this cell** if you want to read directly from Drive. Run it only if you want faster I/O — it copies the dataset to Colab's local SSD once (~5-15 min).

In [ ]:
import os, shutil

DRIVE_NABIRDS = '/content/drive/MyDrive/nabirds'   # folder on Drive
LOCAL_NABIRDS = '/content/nabirds'                  # Colab local disk

if not os.path.exists(LOCAL_NABIRDS):
    print('Copying... (this may take a few minutes)')
    shutil.copytree(DRIVE_NABIRDS, LOCAL_NABIRDS)
    print('Done!')
else:
    print('Already copied, skipping.')

In [ ]:
import os

# --- Paths: read directly from Drive ---
DRIVE_BASE  = '/content/drive/MyDrive/nabirds'
TRAIN_CSV   = os.path.join(DRIVE_BASE, 'train.csv')
VAL_CSV     = os.path.join(DRIVE_BASE, 'val.csv')
IMG_DIR     = os.path.join(DRIVE_BASE, 'images')
SAVE_DIR    = os.path.join(DRIVE_BASE, 'checkpoints')
CHECKPOINT_PATH = os.path.join(SAVE_DIR, 'training_checkpoint.pth')

# If you ran the optional copy cell above, use this instead:
# DRIVE_BASE = '/content/nabirds'

os.makedirs(SAVE_DIR, exist_ok=True)

# Verify
print('train.csv exists:', os.path.exists(TRAIN_CSV))
print('val.csv   exists:', os.path.exists(VAL_CSV))
print('images    exists:', os.path.exists(IMG_DIR))
print('checkpoint path :', CHECKPOINT_PATH)

## 3. Dependencies

In [ ]:
!pip install -q tqdm torchvision

## 4. Dataset & DataLoader

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd

IMG_SIZE = 192
BATCH_SIZE = 64

def build_train_transforms(img_size=224):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

def build_val_transforms(img_size=224):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

class BirdDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.metadata = pd.read_csv(csv_file)
        self.img_dir  = os.path.abspath(img_dir)
        self.transform = transform
        self.gender_map = {'Male': 0, 'Female': 1, 'Unknown': 2}
        self.age_map    = {'Adult': 0, 'Juvenile': 1, 'Unknown': 2}

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.metadata.iloc[idx]['file_path'])
        image = Image.open(img_path).convert('RGB')

        species_label = int(self.metadata.iloc[idx]['class_id'])
        gender_label  = self.gender_map[self.metadata.iloc[idx]['gender']]
        age_label     = self.age_map[self.metadata.iloc[idx]['age']]

        if self.transform:
            image = self.transform(image)

        return image, {
            'species': torch.tensor(species_label, dtype=torch.long),
            'gender':  torch.tensor(gender_label,  dtype=torch.long),
            'age':     torch.tensor(age_label,     dtype=torch.long)
        }

NUM_WORKERS = max(2, min(8, (os.cpu_count() or 2) - 1))
PREFETCH_FACTOR = 2

train_dataset = BirdDataset(TRAIN_CSV, IMG_DIR, transform=build_train_transforms(IMG_SIZE))
val_dataset   = BirdDataset(VAL_CSV,   IMG_DIR, transform=build_val_transforms(IMG_SIZE))

common_loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': torch.cuda.is_available(),
    'persistent_workers': NUM_WORKERS > 0,
}
if NUM_WORKERS > 0:
    common_loader_kwargs['prefetch_factor'] = PREFETCH_FACTOR

train_loader = DataLoader(train_dataset, shuffle=True, **common_loader_kwargs)
val_loader   = DataLoader(val_dataset,   shuffle=False, **common_loader_kwargs)

print(f'Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')
print(f'IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, NUM_WORKERS={NUM_WORKERS}')

## 5. Model

In [ ]:
import torch.nn as nn
import torchvision.models as models

class MultiHeadBirdModel(nn.Module):
    def __init__(self, num_species=555):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights='DEFAULT')
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()

        self.species_head = nn.Sequential(nn.Dropout(0.2), nn.Linear(num_features, num_species))
        self.gender_head  = nn.Sequential(nn.Dropout(0.2), nn.Linear(num_features, 3))
        self.age_head     = nn.Sequential(nn.Dropout(0.2), nn.Linear(num_features, 3))

    def forward(self, x):
        features = self.backbone(x)
        return {
            'species': self.species_head(features),
            'gender':  self.gender_head(features),
            'age':     self.age_head(features)
        }

model = MultiHeadBirdModel(num_species=555).to(device)
print('Model ready.')

## 6. Training (AMP + Auto Resume)

Uses mixed precision for speed and saves a checkpoint after every epoch.
If runtime disconnects, training continues automatically from `training_checkpoint.pth`.

In [ ]:
import torch.optim as optim
from tqdm.notebook import tqdm

if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True

EPOCHS = 30
LR     = 1e-4
amp_enabled = (device.type == 'cuda')

optimizer  = optim.AdamW(model.parameters(), lr=LR)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler     = torch.cuda.amp.GradScaler(enabled=amp_enabled)

criterion_species = nn.CrossEntropyLoss()
criterion_gender  = nn.CrossEntropyLoss()
criterion_age     = nn.CrossEntropyLoss()

def evaluate(model, loader, amp_enabled=False):
    model.eval()
    correct_s = correct_g = correct_a = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(device, non_blocking=True)
            targets = {k: v.to(device) for k, v in labels.items()}
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                outputs = model(images)
            _, pred_s = torch.max(outputs['species'], 1)
            _, pred_g = torch.max(outputs['gender'],  1)
            _, pred_a = torch.max(outputs['age'],     1)
            total     += targets['species'].size(0)
            correct_s += (pred_s == targets['species']).sum().item()
            correct_g += (pred_g == targets['gender']).sum().item()
            correct_a += (pred_a == targets['age']).sum().item()
    return (100 * correct_s / total, 100 * correct_g / total, 100 * correct_a / total)

history  = {'train_loss': [], 'val_accs': {'species': [], 'gender': [], 'age': []}}
best_acc = 0.0
start_epoch = 0

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint.get('scaler_state_dict', scaler.state_dict()))
    history = checkpoint.get('history', history)
    best_acc = checkpoint.get('best_acc', 0.0)
    start_epoch = checkpoint.get('epoch', -1) + 1
    print(f'Resuming from epoch {start_epoch + 1}/{EPOCHS}')

for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for images, labels in pbar:
        images  = images.to(device, non_blocking=True)
        targets = {k: v.to(device) for k, v in labels.items()}
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            outputs    = model(images)
            loss_s     = criterion_species(outputs['species'], targets['species'])
            loss_g     = criterion_gender(outputs['gender'], targets['gender'])
            loss_a     = criterion_age(outputs['age'], targets['age'])
            total_loss = loss_s + 0.5 * loss_g + 0.5 * loss_a

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += total_loss.item()
        pbar.set_postfix({'loss': f'{running_loss / (pbar.n + 1):.4f}'})

    avg_loss = running_loss / len(train_loader)
    s_acc, g_acc, a_acc = evaluate(model, val_loader, amp_enabled=amp_enabled)

    history['train_loss'].append(avg_loss)
    history['val_accs']['species'].append(s_acc)
    history['val_accs']['gender'].append(g_acc)
    history['val_accs']['age'].append(a_acc)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_acc': best_acc,
        'history': history,
    }, CHECKPOINT_PATH)

    print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | LR: {current_lr:.2e}')
    print(f'  Val -> Species: {s_acc:.2f}% | Gender: {g_acc:.2f}% | Age: {a_acc:.2f}%')
    print(f'  --> Checkpoint saved: {CHECKPOINT_PATH}')

    if s_acc > best_acc:
        best_acc  = s_acc
        save_path = os.path.join(SAVE_DIR, 'best_bird_model.pth')
        torch.save(model.state_dict(), save_path)
        print(f'  --> New best model saved! ({save_path})')

print('\nTraining complete!')

## 7. Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], marker='o')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True)

for task, accs in history['val_accs'].items():
    axes[1].plot(accs, marker='o', label=task.capitalize())
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, 'training_plots.png')
plt.savefig(plot_path)
plt.show()
print(f'Plot saved: {plot_path}')

## 8. Download Model (optional)

In [ ]:
# Model is already saved to Drive. Run this to download it directly to your computer:
from google.colab import files
files.download(os.path.join(SAVE_DIR, 'best_bird_model.pth'))